# On-device continual triage with **online SDFT** (LFM2.5-230M)

A phone-hosted 230M model learns your **drifting** attention policy — *should this
notification INTERRUPT me now, or wait for the DIGEST?* — from **implicit feedback**
(did you open it now, or let it wait), **no gold labels**, one `batch_size=1` LoRA
step at a time. The learned policy lives in a **~2.8 MB adapter**, so serving is a
bare ~80-token prompt: no growing in-context cheat-sheet (ICL), no retrieval index
(RAG).

**The twist:** halfway through the stream you go **on-call**, so the policy *drifts*
— monitoring alerts flip `DIGEST → INTERRUPT`, non-incident manager pings flip
`INTERRUPT → DIGEST`. All three "real" methods can reach the new policy — but ICL
and RAG only by paying a **5× token tax on every notification** (and lugging your
whole labeled history around); online SDFT bakes it into weights and serves it free.

**Standalone** — no `git clone`, no package import. Paper:
[Shenfeld et al., 2026](https://self-distillation.github.io/SDFT)
([arXiv](https://arxiv.org/abs/2601.19897)). Base:
[Liquid AI LFM2.5-230M](https://huggingface.co/LiquidAI/LFM2.5-230M).

Arms compared on held-out items under the **current** policy:
- **ZS** — bare prompt, base priors
- **ICL k** — k current-policy demos prepended every call (token tax; needs labels)
- **RAG k** — k nearest past decisions retrieved from a store (5× tokens + a disk index)
- **Online-SDFT** — streamed `batch_size=1` updates from implicit feedback; served bare


## Setup
Colab: **Runtime → GPU** (T4 is plenty). Installs deps and drops stale `torchao`
so PEFT LoRA doesn't hit a version gate.

In [ ]:
%pip install -q "transformers>=4.54" "trl>=0.19" "peft>=0.15" "datasets>=2.19" "accelerate>=0.33" matplotlib
%pip uninstall -y -q torchao
print("setup ok — online SDFT triage demo")

## Config + a tiny drifting inbox

The policy is a plain function of `(category, phase)`. Category mix is **class-balanced**
per phase so a tiny adapter learns the *mapping*, not the majority class. The vocabulary
trap: `receipt` ("your **payment** succeeded") and `monitoring` ("**payment** latency")
share words but flip to opposite actions — so this is a task you must learn the *policy*
for, not the keywords (watch zero-shot walk right into it).

In [ ]:
import random, re
import torch
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "LiquidAI/LFM2.5-230M"
SEED = 7
ACTIONS = ("INTERRUPT", "DIGEST")
STREAM_LEN, DRIFT_AT, EVAL_N = 40, 20, 12
CHECKPOINTS = (5, 10, 15, 20, 25, 30, 35, 40)
LORA_R, LORA_ALPHA = 16, 32
LORA_TARGET = r".*self_attn\.(q|k|v|out)_proj"
LR, MAX_LEN, MAX_NEW = 1e-3, 512, 8   # persistent-optimizer online loop (1-2 token completions)
ICL_K = RAG_K = 4
TEACHER_SHOTS, REPLAY, STEPS_PER_ITEM = 2, 16, 3

NAMES = ["Priya", "Marcus", "Lena", "Diego", "Aisha", "Tom", "Yuki", "Sam"]
PROJECTS = ["Atlas", "the Q3 launch", "the billing rewrite", "Project Nomad", "the search revamp"]
PRODUCTS = ["Nimbus", "the mobile app", "SoundOff", "Kettle", "Zephyr"]
INCIDENT = ["payment latency", "checkout 5xx errors", "a pager alert", "an API outage",
            "elevated error rate", "payment gateway timeouts", "a DB failover"]

def _pick(rng, xs): return xs[rng.randrange(len(xs))]

def gen_item(cat, rng):
    if cat == "mgr_project":
        p = _pick(rng, PROJECTS)
        return {"channel":"email","sender":f"{_pick(rng,NAMES)} (your manager)",
                "subject":f"Need your call on {p}",
                "snippet":_pick(rng,[f"Can you weigh in before standup? Blocking {p}.",
                                     f"Quick decision on {p} — reviewers waiting on you.",
                                     f"Are we shipping {p} today? Need your sign-off."]),"category":cat}
    if cat == "teammate_fyi":
        return {"channel":"slack","sender":f"{_pick(rng,NAMES)} (teammate)","subject":"fyi / no rush",
                "snippet":_pick(rng,["Left comments on your PR whenever.","Shared sync notes — no action today.",
                                     "Maybe refactor utils sometime, curious what you think."]),"category":cat}
    if cat == "calendar_soon":
        w=_pick(rng,NAMES)
        return {"channel":"calendar","sender":"Calendar","subject":f"Starts in 20 min: 1:1 with {w}",
                "snippet":f"Reminder: your meeting with {w} begins soon.","category":cat}
    if cat == "promo":
        return {"channel":"email","sender":f"{_pick(rng,PRODUCTS)} Team",
                "subject":_pick(rng,["48-hour sale — 40% off","New features you'll love","We miss you! 20% off"]),
                "snippet":"Limited time only. Unsubscribe anytime.","category":cat}
    if cat == "social":
        return {"channel":"push","sender":"Social",
                "subject":_pick(rng,["5 people liked your post","3 new followers","Someone mentioned you"]),
                "snippet":"Tap to see the activity.","category":cat}
    if cat == "receipt":
        pr=_pick(rng,PRODUCTS)
        return {"channel":"email","sender":"Receipts","subject":f"Your payment to {pr} was successful",
                "snippet":_pick(rng,["Payment of $12.00 received. Thanks!","Invoice attached. No action required.",
                                     "Charge confirmed. View billing anytime."]),"category":cat}
    if cat == "monitoring":
        s=_pick(rng,INCIDENT)
        return {"channel":"system","sender":"Monitoring Bot","subject":f"ALERT: {s}",
                "snippet":_pick(rng,[f"Automated alert: {s} detected in production.",
                                     f"Threshold breached — {s}. Dashboard inside.",
                                     f"{s} on the checkout service. Auto-generated."]),"category":cat}

def true_action(cat, phase):
    base = {"mgr_project":"INTERRUPT","calendar_soon":"INTERRUPT","teammate_fyi":"DIGEST",
            "promo":"DIGEST","social":"DIGEST","receipt":"DIGEST","monitoring":"DIGEST"}
    if phase == 1: return base[cat]
    d = dict(base); d["monitoring"]="INTERRUPT"; d["mgr_project"]="DIGEST"; return d[cat]

P1 = {"mgr_project":6,"calendar_soon":4,"monitoring":4,"teammate_fyi":2,"promo":2,"social":1,"receipt":1}
P2 = {"monitoring":6,"calendar_soon":4,"mgr_project":6,"teammate_fyi":2,"promo":1,"social":1}
EV = {"monitoring":3,"calendar_soon":3,"mgr_project":3,"teammate_fyi":1,"promo":1,"social":1}

def _spec(spec, phase, rng):
    xs=[]
    for c,n in spec.items():
        for _ in range(n):
            it=gen_item(c,rng); it["phase"]=phase; it["action"]=true_action(c,phase); xs.append(it)
    rng.shuffle(xs); return xs

def build_stream(rng): return _spec(P1,1,rng)+_spec(P2,2,rng)
def build_eval(rng,phase): return _spec(EV,phase,rng)

def item_block(it):
    return f"Channel: {it['channel']}\nFrom: {it['sender']}\nSubject: {it['subject']}\n{it['snippet']}"
def render_prompt(it):
    return ("Triage this inbox item. Should it buzz the user now or wait for the digest? "
            "Answer with exactly one of INTERRUPT or DIGEST.\n\n"+item_block(it))
print("config + data ready")

## Model + pipeline helpers (inlined)

In [ ]:
def device():
    if torch.cuda.is_available(): return "cuda"
    if getattr(torch.backends,"mps",None) and torch.backends.mps.is_available(): return "mps"
    return "cpu"
DEV = device(); print("device:", DEV)

def load_tok():
    t = AutoTokenizer.from_pretrained(MODEL_NAME)
    if t.pad_token is None: t.pad_token = t.eos_token
    t.padding_side = "left"; return t
def load_base():
    dt = torch.float16 if DEV=="cuda" else torch.float32
    m = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=dt)
    return m if DEV=="cuda" else m.to(DEV)
def to_dev(enc, model):
    d = next(model.parameters()).device
    return {k:(v.to(d) if torch.is_tensor(v) else v) for k,v in enc.items()}

def demo_msg(it, a): return [{"role":"user","content":render_prompt(it)},{"role":"assistant","content":a}]
def build_msgs(it, demos=None):
    m=[];
    for d,a in demos or []: m += demo_msg(d,a)
    m.append({"role":"user","content":render_prompt(it)}); return m

ACTION_RE = re.compile(r"\b(INTERRUPT|DIGEST)\b", re.I)
def parse_action(t):
    m = ACTION_RE.search(t or ""); return m.group(1).upper() if m else "NONE"
def prompt_tokens(tok, msgs):
    txt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    return len(tok(txt, add_special_tokens=False)["input_ids"])
def accuracy(items, gens):
    return sum(parse_action(g)==it["action"] for it,g in zip(items,gens))/max(len(items),1)

@torch.inference_mode()
def generate(model, tok, items, demos_fn, label="gen", bs=8, max_new=MAX_NEW):
    model.eval(); outs=[]
    for s in range(0,len(items),bs):
        batch=items[s:s+bs]
        txts=[tok.apply_chat_template(demos_fn(it),tokenize=False,add_generation_prompt=True) for it in batch]
        enc=to_dev(tok(txts,return_tensors="pt",padding=True,add_special_tokens=False),model)
        out=model.generate(**enc,max_new_tokens=max_new,do_sample=False,pad_token_id=tok.pad_token_id)
        for t in tok.batch_decode(out[:,enc["input_ids"].shape[1]:],skip_special_tokens=True):
            outs.append(t.strip())
    return outs

def retriever(store):
    toks=[set(re.findall(r"\w+",render_prompt(s).lower())) for s in store]
    def r(it,k):
        q=set(re.findall(r"\w+",render_prompt(it).lower()))
        order=sorted(range(len(store)),key=lambda i:-len(q&toks[i]))
        return [(store[i],store[i]["action"]) for i in order[:k]]
    return r

# --- persistent-optimizer online updater (Adam momentum carries across the stream) ---
def make_updater(model, tok):
    trainable = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(trainable, lr=LR)
    eos = tok.eos_token or ""
    dev = next(model.parameters()).device
    def step(rows, steps):
        model.train(); model.config.use_cache = False
        for k in range(steps):
            r = rows[k % len(rows)]
            ptxt = tok.apply_chat_template([{"role":"user","content":r["prompt"]}],
                                           tokenize=False, add_generation_prompt=True)
            ids_p = tok(ptxt, add_special_tokens=False)["input_ids"]
            ids_c = tok(r["target"] + eos, add_special_tokens=False)["input_ids"]
            ids = torch.tensor([ids_p + ids_c], device=dev)
            labels = torch.tensor([[-100]*len(ids_p) + ids_c], device=dev)  # completion-only loss
            model(input_ids=ids, labels=labels).loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable, 1.0)  # keep bs=1 steps from diverging
            opt.step(); opt.zero_grad()
        model.config.use_cache = True; model.eval()
    return step
print("helpers ready")

## Build the drifting stream + held-out eval sets

In [ ]:
stream = build_stream(random.Random(SEED))
eval_p1 = build_eval(random.Random(SEED+1), 1)
eval_p2 = build_eval(random.Random(SEED+2), 2)   # current policy after drift
from collections import Counter
print("phase-1 actions:", dict(Counter(it["action"] for it in stream[:DRIFT_AT])))
print("phase-2 actions:", dict(Counter(it["action"] for it in stream[DRIFT_AT:])))
print("\ndrift example — SAME monitoring alert, opposite label:")
mon = [it for it in stream if it["category"]=="monitoring"]
print("  phase 1:", true_action("monitoring",1), "| phase 2:", true_action("monitoring",2))
print("\nsample item:\n", render_prompt(stream[DRIFT_AT])[:200])

## Run it: baselines → self-distillation loop → checkpoints

One base load covers everything. Baselines (ZS/ICL/RAG) are read off the clean base;
then we attach one LoRA adapter and stream `batch_size=1` updates, evaluating on the
held-out **old** and **new** policy sets at each checkpoint so we can watch the drift.

In [ ]:
tok = load_tok(); base = load_base()
phase2 = [it for it in stream if it["phase"]==2]
def _first(c): return next(it for it in phase2 if it["category"]==c)
icl_demos = [(_first(c),_first(c)["action"]) for c in ("monitoring","calendar_soon","mgr_project","teammate_fyi")][:ICL_K]
retrieve = retriever(stream)

def acc_arm(items, demos_fn): return accuracy(items, generate(base,tok,items,demos_fn,bs=8))
print("baselines on current policy (eval_p2)…")
arms = {}
arms["ZS"]          = dict(acc=acc_arm(eval_p2, lambda it: build_msgs(it)),
                           tok=sum(prompt_tokens(tok,build_msgs(it)) for it in eval_p2)/EVAL_N, labels=0)
arms[f"ICL k={ICL_K}"] = dict(acc=acc_arm(eval_p2, lambda it: build_msgs(it, icl_demos)),
                           tok=sum(prompt_tokens(tok,build_msgs(it,icl_demos)) for it in eval_p2)/EVAL_N, labels=ICL_K)
arms[f"RAG k={RAG_K}"] = dict(acc=acc_arm(eval_p2, lambda it: build_msgs(it, retrieve(it,RAG_K))),
                           tok=sum(prompt_tokens(tok,build_msgs(it,retrieve(it,RAG_K))) for it in eval_p2)/EVAL_N, labels=STREAM_LEN)
for k,v in arms.items(): print(f"  {k:10s} acc={v['acc']:.2f} tok/q={v['tok']:.0f}")

# self-distillation targets: the model's OWN guess (conditioned on a couple of recent
# decisions), supervised only by the observed action — reinforce if right, correct if wrong.
def guess_demos(i): return [(stream[j],stream[j]["action"]) for j in range(max(0,i-TEACHER_SHOTS),i)]
guesses = generate(base, tok, list(range(len(stream))),
                   lambda i: build_msgs(stream[i], guess_demos(i)), bs=8)
rows = [{"prompt":render_prompt(it),"target":it["action"],"action":it["action"],
         "feedback":"reinforce" if parse_action(g)==it["action"] else "correct"}
        for it,g in zip(stream,guesses)]
print("\nreinforced (model already right):", sum(r["feedback"]=="reinforce" for r in rows), "/", len(rows))

# stream bs=1 updates with a sliding balanced replay buffer; checkpoint eval
model = get_peft_model(base, LoraConfig(r=LORA_R,lora_alpha=LORA_ALPHA,lora_dropout=0.05,
                                        target_modules=LORA_TARGET,task_type="CAUSAL_LM"))
update = make_updater(model, tok)
curve = {"pos":[], "old":[], "new":[]}
buf=[]; rr=random.Random(SEED)
for i,row in enumerate(rows):
    buf=(buf+[row])[-REPLAY:]
    others=[b for b in buf[:-1] if b["action"]!=row["action"]] or buf[:-1]
    update([row]+(rr.sample(others,1) if others else []), STEPS_PER_ITEM)
    if (i+1) in CHECKPOINTS:
        a1=accuracy(eval_p1, generate(model,tok,eval_p1,lambda it:build_msgs(it),bs=8))
        a2=accuracy(eval_p2, generate(model,tok,eval_p2,lambda it:build_msgs(it),bs=8))
        curve["pos"].append(i+1); curve["old"].append(a1); curve["new"].append(a2)
        print(f"  streamed {i+1:2d}: old-policy acc={a1:.2f}  new-policy acc={a2:.2f}")
arms["Online-SDFT"] = dict(acc=curve["new"][-1], tok=arms["ZS"]["tok"], labels=0)
print("\nfinal:", {k:round(v['acc'],2) for k,v in arms.items()})

## The figure: same accuracy, a fraction of the cost — and it survives the drift

In [ ]:
import matplotlib.pyplot as plt
colors={"ZS":"#9aa0a6","ICL":"#e8710a","RAG":"#d93025","Online-SDFT":"#1a73e8"}
def col(n):
    for k,c in colors.items():
        if n.startswith(k): return c
    return "#5f6368"
fig,(axA,axB)=plt.subplots(1,2,figsize=(12.4,4.9))
for n,d in arms.items():
    axA.scatter(d["tok"],d["acc"]*100,s=170,color=col(n),zorder=3,edgecolor="white",linewidth=1.5)
    axA.annotate(n,(d["tok"],d["acc"]*100),textcoords="offset points",
                 xytext=(8, -5 if n.startswith("RAG") else 4),fontsize=10.5,fontweight="bold",color=col(n))
axA.set_xlabel("Recurring prompt tokens / query  (paid on every notification)")
axA.set_ylabel("Accuracy on current policy (%)"); axA.set_ylim(0,105); axA.grid(alpha=.25)
axA.axvspan(0,70,color="#1a73e8",alpha=.05)
axA.set_title("A.  Same accuracy, a fraction of the cost",fontweight="bold")
axB.axvline(DRIFT_AT,color="#5f6368",ls="--",lw=1.3)
axB.text(DRIFT_AT-.6,6,"policy drift\n(you go on-call)",fontsize=8.5,color="#5f6368",ha="right")
axB.plot(curve["pos"],[v*100 for v in curve["old"]],"-o",color="#7b3fa0",lw=2.2,label="old policy (focus week)")
axB.plot(curve["pos"],[v*100 for v in curve["new"]],"-o",color="#1a73e8",lw=2.4,label="new policy (on-call)")
ref=max(arms[f"ICL k={ICL_K}"]["acc"],arms[f"RAG k={RAG_K}"]["acc"])*100
axB.axhline(ref,color="#e8710a",ls=":",lw=1.6,label="ICL / RAG on new policy\n(but +5× tokens every call)")
axB.set_xlabel("Items streamed (one batch_size=1 update each)"); axB.set_ylabel("Held-out accuracy (%)")
axB.set_ylim(0,105); axB.grid(alpha=.25); axB.legend(fontsize=8,loc="lower right",framealpha=.95)
axB.set_title("B.  It learns the new policy from the stream — free at serve time",fontweight="bold")
fig.suptitle(f"On-device continual triage · LFM2.5-230M · policy in a ~{2.76:.1f} MB LoRA adapter, no gold labels",
             fontweight="bold",y=1.02)
fig.tight_layout(); plt.show()

## One item, four minds
The same drifted **monitoring** alert (correct answer after going on-call: `INTERRUPT`).

In [ ]:
q = next(it for it in eval_p2 if it["category"]=="monitoring")
print("PROMPT:\n", render_prompt(q), "\n\ncorrect (on-call):", q["action"], "\n")
with model.disable_adapter():                       # frozen base (adapter OFF) for ZS/ICL/RAG
    zs  = parse_action(generate(model,tok,[q],lambda it:build_msgs(it),bs=1)[0])
    icl = parse_action(generate(model,tok,[q],lambda it:build_msgs(it,icl_demos),bs=1)[0])
    rag = parse_action(generate(model,tok,[q],lambda it:build_msgs(it,retrieve(it,RAG_K)),bs=1)[0])
sdft = parse_action(generate(model,tok,[q],lambda it:build_msgs(it),bs=1)[0])   # adapter ON
extra = arms[f"ICL k={ICL_K}"]["tok"] - arms["ZS"]["tok"]
print("  ZS          ->", zs)
print(f"  ICL k={ICL_K}      ->", icl, f"  (+{extra:.0f} tokens every call)")
print(f"  RAG k={RAG_K}      ->", rag)
print("  Online-SDFT ->", sdft, "  (bare prompt)")

## What to try next
1. **Move the drift** — change `DRIFT_AT`, or add a *third* regime (e.g. weekend).
2. **Make feedback noisier** — flip a fraction of observed actions before training; watch robustness.
3. **Widen the policy** — add categories, or go 3-way (INTERRUPT / LATER / ARCHIVE) and watch a 230M struggle with class balance at `batch_size=1`.
4. **Full toolkit** — [Local-SDFT](https://github.com/lin826/Local-SDFT): the `/data` live chat loop (tone as implicit feedback), GRPO, and the CLI SDFT pipeline.

230M is small enough that curiosity, not VRAM, is the bottleneck.